# Fingerprint Recognition Accuracy Upgrade (SOCOFing + ArcFace)

Fine-tunes the fingerprint embedding model used by `models/fingerprint/inference.py` (ResNet50 + upgraded projection head + ArcFace, 512-dim embedding - see `models/fingerprint/model.py` and `docs/ARCHITECTURE.md`), evaluates it on a genuinely **subject-disjoint** held-out test set, and runs a genuine-vs-impostor demo.

**This supersedes the previous fingerprint checkpoint** (256-dim, a different architecture) - the previous training notebook split each subject's own images 70/15/15, so every subject appeared in train, val, *and* test (identity leakage for a verification task). `models/fingerprint/dataset.py::subject_disjoint_split` fixes this: every subject belongs to exactly one split.

**Dataset:** [SOCOFing](https://www.kaggle.com/datasets/ruizgara/socofing) (Sokoto Coventry Fingerprint Dataset) - 6,000 fingerprints from 600 subjects. See `docs/DATASETS.md`.

## 1. Setup

In [ ]:
# --- Kaggle Kernel setup ---
# GPU + internet are enabled in kernel-metadata.json; the dataset is attached
# natively (see dataset_sources) rather than downloaded manually.
import os, sys
REPO_URL = "https://github.com/Malik8122/Cancelable-Multimodal-Biometric-Authentication-for-Critical-Infrastructure.git"
REPO_DIR = "/kaggle/working/repo"
REPO_BRANCH = "phase-1-foundation"

if not os.path.isdir(REPO_DIR):
    !git clone -q -b $REPO_BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# h5py/albumentations are project dependencies not necessarily preinstalled
# on the base Kaggle image.
!pip install -q h5py albumentations

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Some hosted GPU sessions intermittently hand out a driver/torch-build
# combination where CUDA reports available but no kernel image exists for
# the actual device (observed on this project's Voice-modality Kaggle runs) -
# smoke-test with a real op now and fall back to CPU rather than crashing deep
# into training on a broken GPU. models/fingerprint/utils.py::detect_device()
# does the same check when train() is called directly (e.g. from a plain
# script), so this is redundant-but-harmless belt-and-suspenders here.
if DEVICE == "cuda":
    try:
        (torch.zeros(1, device=DEVICE) + 1).cpu()
    except Exception as e:  # torch.AcceleratorError, RuntimeError, etc.
        print(f"CUDA smoke test failed ({e}); falling back to CPU.")
        DEVICE = "cpu"
print("Using device:", DEVICE)

## 2. Dataset

Attached natively via `dataset_sources` in `kernel-metadata.json` (`ruizgara/socofing`) - no manual download or Kaggle API token needed inside the kernel itself.

In [ ]:
DATASET_ROOT = "/kaggle/input/socofing"
assert os.path.isdir(DATASET_ROOT), f"Expected dataset at {DATASET_ROOT} - check dataset_sources in kernel-metadata.json"

## 3. Build train/val/test datasets (subject-disjoint)

`SocofingDataset` (`models/fingerprint/dataset.py`) discovers every real, unaltered fingerprint image, then `subject_disjoint_split` assigns each *subject* entirely to exactly one split (70% / 15% / 15%, seed 42) - never splitting within a subject. Validation/test subjects are therefore unseen during training; their labels are raw subject ids, not train-time class indices (see the dataset module's docstring for why). This cell is also this project's actual documentation of dataset composition, printed at run time from the real attached data.

In [ ]:
from models.fingerprint.config import FingerprintConfig
from models.fingerprint.dataset import SocofingDataset

config = FingerprintConfig()  # see models/fingerprint/config.py for every hyperparameter default

train_dataset = SocofingDataset(DATASET_ROOT, mode="train", config=config)
val_dataset = SocofingDataset(DATASET_ROOT, mode="val", config=config)
test_dataset = SocofingDataset(DATASET_ROOT, mode="test", config=config)

print(f"Train subjects: {train_dataset.num_classes} | Train images: {len(train_dataset)}")
print(f"Val images:   {len(val_dataset)} (subjects unseen during training)")
print(f"Test images:  {len(test_dataset)} (subjects unseen during training)")

## 4. Train

Calls `models/fingerprint/train.py::train()` directly - the training loop (AdamW + warmup/cosine schedule + AMP + gradient clipping + label-smoothed ArcFace with hard-negative mining after epoch 10 + balanced P/K batch sampling + early stopping on validation EER) lives there, not duplicated in this notebook. Saves `best_model.pt` (lowest EER) / `last_model.pt` every epoch, `metrics.csv`, `training_history.json`, then the canonical `fingerprint_embedder.pt` + `.h5` + `fingerprint_config.json`.

In [ ]:
from models.fingerprint.train import train

OUTPUT_DIR = f"{REPO_DIR}/models/fingerprint/saved"
CHECKPOINT_PATH = train(DATASET_ROOT, OUTPUT_DIR, config=config, device=DEVICE)
print("Saved checkpoint:", CHECKPOINT_PATH)

## 5. Evaluate on the held-out (subject-disjoint) test set

Mirrors `evaluation/experiments.py::run_modality_experiment` (used by Face/Iris/Fingerprint's Phase 1 evaluation) via the fingerprint-specific `evaluation/fingerprint_metrics.py::run_fingerprint_experiment`, which adds precision/recall/F1/confusion-matrix/threshold-sweep on top of the shared FAR/FRR/EER/ROC-AUC primitives. Target: 90-95% accuracy, ROC-AUC 0.97+, FAR/FRR below 5%, EER below 0.05.

In [ ]:
from models.fingerprint.inference import FingerprintEmbedder
from evaluation.fingerprint_metrics import (
    run_fingerprint_experiment, save_confusion_matrix_csv, save_fingerprint_metrics_csv,
    save_roc_csv, save_threshold_sweep_csv, threshold_sweep_table,
)

fine_tuned_embedder = FingerprintEmbedder(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)
assert not fine_tuned_embedder.mock_mode, "Checkpoint failed to load - check the path above."

test_embeddings, test_labels = [], []
for index in range(len(test_dataset)):
    tensor, subject_id = test_dataset[index]
    image_hwc = tensor.permute(1, 2, 0).numpy()  # (3, 224, 224) -> (224, 224, 3), already ImageNet-normalized
    test_embeddings.append(fine_tuned_embedder.extract_embedding(image_hwc))
    test_labels.append(subject_id)

report = run_fingerprint_experiment(test_embeddings, test_labels)
print(f"EER: {report['eer']:.4f} | Accuracy: {report['accuracy']:.4f} | AUC: {report['auc']:.4f}")
print(f"FAR: {report['far']:.4f} | FRR: {report['frr']:.4f}")
print(f"Precision: {report['precision']:.4f} | Recall: {report['recall']:.4f} | F1: {report['f1']:.4f}")

RESULTS_DIR = f"{REPO_DIR}/evaluation/results"
save_fingerprint_metrics_csv(report, f"{RESULTS_DIR}/fingerprint_metrics.csv")
save_roc_csv(report, f"{RESULTS_DIR}/fingerprint_roc.csv")
save_confusion_matrix_csv(report, f"{RESULTS_DIR}/fingerprint_confusion_matrix.csv")
sweep = threshold_sweep_table(report["genuine_scores"], report["impostor_scores"])
save_threshold_sweep_csv(sweep, f"{RESULTS_DIR}/fingerprint_threshold_sweep.csv")
print("Saved metrics CSVs under", RESULTS_DIR)

## 6. Demo: genuine vs. impostor pair

Same qualitative check the other modalities' notebooks end with - two images of the *same* (test-set, never-trained-on) subject should score high, two different subjects should score low.

In [ ]:
import numpy as np
from evaluation.metrics import cosine_similarity

rng = np.random.default_rng(42)
test_label_array = np.array(test_labels)

def pick_pair(same_subject: bool):
    for _ in range(200):
        i, j = rng.choice(len(test_embeddings), size=2, replace=False)
        if (test_label_array[i] == test_label_array[j]) == same_subject:
            return i, j
    raise RuntimeError("Could not find a suitable pair - try a larger test split.")

genuine_i, genuine_j = pick_pair(same_subject=True)
impostor_i, impostor_j = pick_pair(same_subject=False)
print("Genuine pair similarity:  ", cosine_similarity(test_embeddings[genuine_i], test_embeddings[genuine_j]))
print("Impostor pair similarity:", cosine_similarity(test_embeddings[impostor_i], test_embeddings[impostor_j]))

## Output

Kaggle automatically persists everything written under `/kaggle/working/` as this kernel's output. After the run finishes, fetch the checkpoints with:
```bash
kaggle kernels output <username>/fingerprint-embedding-training-phase-1 -p ./kaggle_output/fingerprint
```
then copy the `.pt`/`.h5` files from `kaggle_output/fingerprint/repo/models/fingerprint/saved/` into this repo's `models/fingerprint/saved/` and commit them (Git LFS picks up the `.pt`/`.h5` files automatically) - or just run `python scripts/run_kaggle_kernels.py --only fingerprint`, which automates all of this.